In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))


<font size="6" color="red">ch15. 데이터베이스 연동</font>
# 1절. SQLite 데이터 베이스 연결
- SQLite 데이터베이스는 별도의 DBMS없이 SQL을 이용해서 DB액세스할 수 있도록 만든 간단한 디스크기반DB제공
- C라이브러리
- SQLite는 프로토타입을 만들 때 사용 
- 프로젝트단계 : 분석 -> 설계 -> 구현 -> 테스트 -> 고객에게 배포 -> 유지보수
              -> 프로토타입(SQLite) 시제품(구현후 반양산직전) 
              완제품(Oracle, MySQL, MariDB, PostgreSQL, MSSQL, 아마존auroraDB, ...)
- [DB Browser for SQLite](https://sqlitebrowser.org/)에서 "DB Browser for SQLite - .zip (no installer) for 64-bit Windows" 다운로드후 압축풀기

## 1.1 SQLite browser 설치 및 sqlite3패키지 load

In [6]:
import sqlite3
sqlite3.sqlite_version

'3.40.1'

In [7]:
import pandas as pd
pd.__version__

'1.5.3'

## 1.2 데이터베이스 연결
- 데이터베이스연결 객체 -> 커서객체(SQL전송 및 결과 받는 객체) -> 원하는 로직 수행 -> 커서객체 해제 
- > DB연결객체 해제(close)
- SQLite로 DB 연결객체 생성시, DB파일이 있으면 연결, DB파인이 없으면 빈 DB파일 생성

In [8]:
# DB연결 (여기서 에러가 날 경우 VS_redist.x64.exe 설치)
conn = sqlite3.connect("data/ch15_example.db")
conn

In [9]:
# 커서 객체 생성 : 커서는 SQL문 실행시키고, 결과를 받는 객체
cursor = conn.cursor()
cursor

In [10]:
cursor.execute('''
    CREATE TABLE MEMBER (
        NAME TEXT,
        AGE  INT,
        EMAIL TEXT 
    )
''')

OperationalError: table MEMBER already exists

In [ ]:
cursor.execute('DROP TABLE MEMBER')

In [ ]:
sql = 'INSERT INTO MEMBER VALUES (\'홍길동\', 25, \'h@h.com\')'
cursor.execute(sql)
print('insert, update, delete문의 수행 결과 행수 : ', cursor.rowcount)
sql = "INSERT INTO MEMBER VALUES ('신길동', 30, 's@s.com')"
cursor.execute(sql)
print('수행 결과 행수 : ', cursor.rowcount)
sql = "INSERT INTO MEMBER VALUES ('신림동', 35, 'sil@h.com')"
cursor.execute(sql)
print('수행 결과 행수 : ', cursor.rowcount)

In [ ]:
conn.commit() # 反. conn.rollback()DML에서만 commit이나 rollback

In [ ]:
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE") # SELECT sql문 전송

In [ ]:
# INSERT, UPDATE, DELETE문 실행결고 : cursor.rowcount
# SELECT문 실행결과를 받는 함수들 
    # fetchone() : 결과를 한행씩 받을 때 (튜플)
    # fetchall() : 결과를 모두 받을 때 (튜플list)
    # fetchmant(n) : 결과를 n행 받을 때(튜플 list)
cursor.fetchall()

In [ ]:
cursor.fetchall() # 한번 소요된 cursor 객체는 다시 fetch 할 수 없음

In [ ]:
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = cursor.fetchall()
members

In [ ]:
# 한줄씩 읽어서 dict list에 append
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = []
while True: 
    member = cursor.fetchone() # SQL문 수행 결과를 한줄 가져오기
    if member is None:
        break
    members.append({'name':member[0], 'age':member[1], 'email':member[2]})
members

In [ ]:
class Member:
    'Member 테이블의 내용을 받은 객체 타입'
    def __init__(self, name, age, email):
        self.name = name
        self.age = age
        self.email = email
    def __str__(self):
        return "{}\t{}\t{}".format(self.name, self.age, self.email)
m = Member('홍길동', 25, 'h@h.com')
print(m)

In [ ]:
dbmember = ('홍길동', 25, 'h@h.com')
m = Member(*dbmember)
print(m)

In [ ]:
# 한줄씩 읽어서 객체list에 append
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = []
while True:
    dbmember = cursor.fetchone()
    if dbmember is None:
        break
    member = Member(*dbmember)
    members.append(member)
for mem in members:
    print(mem)

In [ ]:
# 최상위 n행 읽어오기
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = cursor.fetchmany(2)
members

In [ ]:
cursor.close()
conn.close()

## 1.3 SQL 구문에 파라미터 사용하기
- qmark(DB에 따라 불가한 경우가 있음)
- named(추천)

In [ ]:
conn = sqlite3.connect('data/ch15_example.db')
cursor = conn.cursor()
cursor.execute("SELECT * FROM MEMBER WHERE NAME IN ('홍길동','신길동')")
cursor.fetchall()

In [ ]:
# 파라미터 사용하기 : qmark 방법 이용
name1 = input('검색할 이름 1 : ')
name2 = input('검색할 이름 2 : ')
# cursor.execute(f"SELECT * FROM MEMBER WHERE NAME IN ('{name1}', '{name2}')")
cursor.execute("SELECT * FROM MEMBER WHERE NAME IN (?, ?)", (name1, name2))
cursor.fetchall()

In [ ]:
# 파라미터 사용하기 : named 방법 이용
name1 = input('검색할 이름 1 : ')
name2 = input('검색할 이름 2 : ')
# cursor.execute(f"SELECT * FROM MEMBER WHERE NAME IN ('{name1}', '{name2}')")
cursor.execute("SELECT * FROM MEMBER WHERE NAME IN (:name1, :name2)", {'name1':name1,
                                                                       'name2':name2})
cursor.fetchall()

In [ ]:
# 파라미터 사용하기 : named 방법 이용
name = input('회원가입할 이름은?')
try:
    age = int(input('나이는 ? (꼭 숫자로) '))
except:
    print('유효하지 않은 나이를 입력할경우 1세로 초기화합니다')
    age = 1
email = input('이메일은?')
cursor.execute("INSERT INTO MEMBER VALUES (:name, :age, :email)",
               {'name':name, 'age':age, 'email':email}) # sql 전송
conn.commit()
print('수행 결과 행수 : ', cursor.rowcount)

In [ ]:
cursor.close()
conn.close()

# 2절. 오라클 데이터베이스 연결
- pip install cx_oracle(11g까지), pip install oracledb(12버전부터)

In [ ]:
import cx_Oracle
cx_Oracle.__version__

In [11]:
# conn 얻어오는 방법1
oracle_dsn = cx_Oracle.makedsn(host="localhost", port=1521, sid='xe')
conn = cx_Oracle.connect(user='scott', password='tiger', dsn=oracle_dsn)
conn.close()

NameError: name 'cx_Oracle' is not defined

In [ ]:
# conn 얻어오는 방법 2 : (여기서 에러가 날 경우 VS_redist.x64.exe 설치)
conn = cx_Oracle.connect('scott', 'tiger', 'localhost:1521/xe')
conn

In [12]:
# cursor 객체 생성하고 sql문 전송&결과 받기
cursor = conn.cursor()
sql = "SELECT EMPNO NO, ENAME, JOB, MGR, HIREDATE, SAL, COMM, DEPTNO FROM EMP"
cursor.execute(sql)
emps = cursor.fetchall()

OperationalError: no such table: EMP

In [ ]:
for emp in emps: 
    print(emp)

In [13]:
cursor.description

In [14]:
[descipt[0] for descipt in cursor.description]

TypeError: 'NoneType' object is not iterable

In [ ]:
import pandas as pd
emp_df = pd.DataFrame(emps,
                      columns=[descipt[0] for descipt in cursor.description])
emp_df

In [15]:
# 사용자로부터 검색할 이름을 받아 해당 데이터 출력
sql = "SELECT * FROM EMP WHERE ENAME=(:ename)"
ename = input('검색할 이름 ? ').upper()
cursor.execute(sql, {'ename':ename})
emp = cursor.fetchone() # 결과가 있으면 해당 데이터를 튜플로, 결과가 없으면 None으로
if emp:
    columns = [descript[0] for descript in cursor.description]
    df = pd.DataFrame([emp], columns=columns)
    display(df)
else:
    print('해당 이름이 없습니다.')

검색할 이름 ? 


OperationalError: no such table: EMP

In [ ]:
for col, data in zip(columns, emp):
    print("{}:{}".format(col, data if data is not None else '-'))

In [16]:
cursor.close()
conn.close()

# 3절. MySQL 연결

| 라이브러리 | 특징 | 
|:--|:--|
| **mysql-connector-python** | MySQL 공식 커넥터. python으로 구현 |
| **pyMySQL** | 커뮤니티에서 만든 서드파트 라이브러리(경량). python으로 구현. 널리 쓰임 |

- pip install pyMySQL

In [17]:
%pip install pyMySQL

Note: you may need to restart the kernel to use updated packages.


In [22]:
import pymysql 
conn = pymysql.connect(
    host='127.0.0.1',
    user='root',
    password='jmk31090~!@',
    database='devdb',
    charset='utf8mb4',
    autocommit=True
)
cursor = conn.cursor()
sql = 'select * from person'
cursor.execute(sql)
person = cursor.fetchall()
print(person)
cursor.close()
conn.close()

((1001, 'bill', 'president', None, datetime.date(1989, 1, 10), Decimal('7000'), None, 10), (1111, 'smith', 'manager', 1001, datetime.date(1990, 12, 12), Decimal('1000'), None, 10), (1112, 'ally', 'salesman', 1116, datetime.date(1991, 2, 20), Decimal('1600'), Decimal('500'), 30), (1113, 'word', 'salesman', 1116, datetime.date(1992, 2, 24), Decimal('1450'), Decimal('300'), 30), (1114, 'james', 'manager', 1001, datetime.date(1990, 4, 12), Decimal('3975'), None, 20), (1116, 'johnson', 'manager', 1001, datetime.date(1991, 5, 1), Decimal('3550'), None, 30), (1118, 'martin', 'analyst', 1111, datetime.date(1991, 9, 9), Decimal('3450'), None, 10), (1121, 'kim', 'clerk', 1114, datetime.date(1990, 12, 8), Decimal('4000'), None, 20), (1123, 'lee', 'salesman', 1116, datetime.date(1991, 9, 23), Decimal('1200'), Decimal('0'), 30), (1226, 'park', 'analyst', 1111, datetime.date(1990, 1, 3), Decimal('2500'), None, 10))


In [30]:
import pandas as pd
import numpy as np
df = pd.DataFrame(person,columns=[descript[0] for descript in cursor.description])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   pno       10 non-null     int64  
 1   pname     10 non-null     object 
 2   job       10 non-null     object 
 3   manager   9 non-null      float64
 4   hiredate  10 non-null     object 
 5   sal       10 non-null     object 
 6   comm      3 non-null      object 
 7   dno       10 non-null     int64  
dtypes: float64(1), int64(2), object(5)
memory usage: 768.0+ bytes


In [32]:
df.isna().sum()

pno         0
pname       0
job         0
manager     1
hiredate    0
sal         0
comm        7
dno         0
dtype: int64

In [36]:
df['hiredate'] = df['hiredate'].astype('datetime64[ns]')

In [38]:
df.astype({'sal':'float64', 'comm':np.float64})
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   pno       10 non-null     int64         
 1   pname     10 non-null     object        
 2   job       10 non-null     object        
 3   manager   9 non-null      float64       
 4   hiredate  10 non-null     datetime64[ns]
 5   sal       10 non-null     object        
 6   comm      3 non-null      object        
 7   dno       10 non-null     int64         
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 768.0+ bytes


In [43]:
cursor.close()
conn.close()

Error: Already closed

# 4절. 연습문제
## oracle연동
- 회원가입 | 전체조회 | 이름찾기 | 메일삭제 | csv내보내기 | 종료
### 0. 처음 실행

In [44]:
def load_conn():
    global conn # 변수를 전역변수로 쓰겠다 
    import cx_Oracle
    conn = cx_Oracle.connect("scott", "tiger", "localhost:1521/xe")
load_conn()

### 1.회원입력

In [6]:
def fn1_insert_member():
    '사용자로부터 이름, 전화, 이메일, 나이, 등급(1~5)을 입력받아 DB에 insert한다'
    # 사용자로부터 데이터 받기
    name = input('이름 : ')
    phone = input('전화 : ')
    email = input('메일 : ')
    try:
        age = int(input('나이 : '))
        if age<0 :
            age = 0
    except:
        print('유효하지 않은 나이 입력시 나이는 0으로 초기화')
        age = 0
    try:
        grade = int(input('등급(1~5) : '))
        if grade < 1:
            grade = 1
        elif grade > 5:
            grade = 5
    except:
        print('유효하지 않은 등급을 입력시 1로 초기화')
        grade = 1
    # SQL 전송 및 결과 받기
    cursor = conn.cursor()
    sql = "INSERT INTO MEMBER VALUES (:name, :phone, :email, :age, :grade)"
    cursor.execute(sql, {'name':name, 
                         'phone':phone, 
                         'email':email, 
                         'age':age, 
                         'grade':grade}) # sql전송 및 결과 받기
    if cursor.rowcount:
        conn.commit()
        print(name + '님 회원가입 완료')
    cursor.close()
# fn1_insert_member() 

### 2. 전체조회


In [7]:
def fn2_display_members():
    'member 테이블의 내용을 데이터프레임으로 display'
    import pandas as pd
    cursor = conn.cursor()
    sql = "SELECT NAME, PHONE, EMAIL, AGE, GRADE FROM MEMBER ORDER BY AGE"
    cursor.execute(sql)
    members = cursor.fetchall() # 데이터가 있으면 튜플 list, 데이터가 없으면 빈 list
    if members:   
        columns = [descript[0] for descript in cursor.description]
        df = pd.DataFrame(members,columns=columns)
        display(df)
    else:
        print('입력된 회원이 없습니다')
    cursor.close()
# fn2_display_members()

### 3. 이름으로 조회

In [8]:
def fn3_search_name():
    '사용자로부터 이름을 입력받아 해당 회원의 정보를 데이터프레임으로 display'
    search_name = input('조회할 이름 : ')
    cursor = conn.cursor()
    sql = "SELECT NAME, PHONE, EMAIL, AGE, GRADE FROM MEMBER WHERE NAME = :name ORDER BY AGE"
    cursor.execute(sql, {'name': search_name})
    members = cursor.fetchall()
    if members:
        columns = [descript[0] for descript in cursor.description]
        df = pd.DataFrame(members, columns=columns)
        display(df)
    else:
        print(f"'{search_name}' 회원을 찾을 수 없습니다.")
    cursor.close()
# fn3_search_name()

### 4. 메일로 삭제

In [9]:
def fn4_delete_member():
    email = input('삭제할 회원의 메일 : ')
    cursor = conn.cursor()
    sql = "DELETE FROM MEMBER WHERE EMAIL = :email"
    cursor.execute(sql, {'email': email})
    if cursor.rowcount:
        conn.commit()
        print(f"[{email}] 회원 정보가 성공적으로 삭제되었습니다.")
    else:
        print(f"[{email}] 메일을 가진 회원이 존재하지 않습니다.")
    cursor.close()
# fn4_delete_member()

### 5. csv 내보내기
- data/ch15_member.csv로 회원정보 내보내기

In [10]:
def fn5_save_csv():
    'member 테이블의 모든 회원 정보를 data/ch15_member.csv 파일로 내보낸다'
    folder_path = 'data'
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
    cursor = conn.cursor()
    sql = "SELECT NAME, PHONE, EMAIL, AGE, GRADE FROM MEMBER ORDER BY AGE"
    cursor.execute(sql)
    members = cursor.fetchall()
    if members:
        columns = [descript[0] for descript in cursor.description]
        df = pd.DataFrame(members, columns=columns)
        file_path = os.path.join(folder_path, 'ch15_member.csv')
        df.to_csv(file_path, index=False, encoding='utf-8-sig') 
        print(f"회원 정보가 성공적으로 '{file_path}' 파일로 내보내졌습니다. (총 {len(df)}명)")
    else:
        print('내보낼 회원 정보가 없습니다.')
    cursor.close()
# fn5_save_csv()

### 6.기능들 합하기

In [ ]:
import os
import pandas as pd
def main():
    while True:
        menu = input("1:입력 | 2.전체조회 | 3.이름찾기 | 4.메일삭제 | 5.csv백업 | 9.종료")
        if menu=='1':
            fn1_insert_member()
        elif menu=='2':
            fn2_display_members()
        elif menu=='3':
            fn3_search_name()
        elif menu=='4':
            fn4_delete_member()
        elif menu=='5':
            fn5_save_csv()
        elif menu=='9':
            conn.close()
            break
        else:
            print('유효한 메뉴 번호를 입력해주세요.')
if __name__=='__main__':
    global conn
    import cx_Oracle
    conn = cx_Oracle.connect("scott", "tiger", "localhost:1521/xe")
    main()
        

1:입력 | 2.전체조회 | 3.이름찾기 | 4.메일삭제 | 5.csv백업 | 9.종료5
회원 정보가 성공적으로 'data\ch15_member.csv' 파일로 내보내졌습니다. (총 4명)
1:입력 | 2.전체조회 | 3.이름찾기 | 4.메일삭제 | 5.csv백업 | 9.종료2


,NAME,PHONE,EMAIL,AGE,GRADE
0,홍길동,010-2233-4455,r@r.com,22,4
1,홍길동,010-9999-9999,h@h.com,26,5
2,홍길동,010-1234-1234,k@k.com,26,5
3,홍길동,010-2222-3333,g@g.com,31,3
